# 06 — Train both Mizo to English translation systems (v2)

Two systems, identical except for the source side:

- **baseline** reads plain Mizo
- **marked** reads Mizo with entity markup from the corrected projection

Both use `mt_v2` data, rebuilt from clean text. Each system is trained in its
own cell and saved immediately, so a failure in one does not cost the other.

**Run from the repository root.** Kernel: `Python (tka)`.
Budget several hours per system; the baseline is the faster of the two.

## Cell 1: Setup

In [6]:
from pathlib import Path
import json, sys, time, gc
import numpy as np
import torch

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
print(f"Repo root: {ROOT}")

DATA   = ROOT / "data" / "processed" / "mt_v2"
MODELS = ROOT / "models"
RES    = ROOT / "results" / "mt"
MODELS.mkdir(exist_ok=True); RES.mkdir(parents=True, exist_ok=True)

need = [f"{sp}_{f}.txt" for sp in ("train","val","test")
        for f in ("mizo_plain","mizo_tagged","english")]
for f in need:
    p = DATA / f
    print(("  ok   " if p.exists() else "  MISS ") + f)
    if not p.exists():
        sys.exit("Run 05_mt_data_regeneration.ipynb first")

print(f"\nPyTorch {torch.__version__}   CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    pr = torch.cuda.get_device_properties(0)
    VRAM = pr.total_memory / 1024**3
    print(f"GPU: {pr.name}   VRAM: {VRAM:.1f} GB")
else:
    VRAM = 0
    sys.exit("No GPU. Translation training on CPU is not practical.")

Repo root: C:\Users\Haulai\mizo-ner
  ok   train_mizo_plain.txt
  ok   train_mizo_tagged.txt
  ok   train_english.txt
  ok   val_mizo_plain.txt
  ok   val_mizo_tagged.txt
  ok   val_english.txt
  ok   test_mizo_plain.txt
  ok   test_mizo_tagged.txt
  ok   test_english.txt

PyTorch 2.7.1+cu118   CUDA: True
GPU: NVIDIA GeForce RTX 3060   VRAM: 12.0 GB


## Cell 2: Load the splits

In [7]:
def lines(p):
    return open(p, encoding="utf-8").read().splitlines()

data = {}
for sp in ("train", "val", "test"):
    data[sp] = {
        "plain":   lines(DATA / f"{sp}_mizo_plain.txt"),
        "tagged":  lines(DATA / f"{sp}_mizo_tagged.txt"),
        "english": lines(DATA / f"{sp}_english.txt"),
    }
    n = {k: len(v) for k, v in data[sp].items()}
    assert len(set(n.values())) == 1, f"{sp}: line counts differ {n}"
    print(f"  {sp:<6}{len(data[sp]['plain']):>8,}")

print(f"\nplain : {data['train']['plain'][0]}")
print(f"tagged: {data['train']['tagged'][0]}")
print(f"target: {data['train']['english'][0]}")

  train  397,060
  val     22,059
  test    22,059

plain : Kar thum chawlh Liana-an a la.
tagged: Kar thum chawlh «PERSON» Liana «/PERSON»-an a la.
target: Liana took three weeks off.


## Cell 3: Source length

Markup adds tokens, so the marked source is longer than the plain source. If
the maximum length truncates marked sentences it would penalize that system for
reasons unrelated to the entity signal. Measure both.

In [8]:
from transformers import AutoTokenizer

MT_MODEL = "Helsinki-NLP/opus-mt-mul-en"
tok = AutoTokenizer.from_pretrained(MT_MODEL)

sample = 20000
for variant in ("plain", "tagged"):
    L = np.array([len(tok(s)["input_ids"]) for s in data["train"][variant][:sample]])
    print(f"{variant:<8} mean {L.mean():5.1f}  median {np.median(L):3.0f}  "
          f"99th {np.percentile(L,99):3.0f}  max {L.max():3.0f}")
    for cand in (128, 192, 256):
        lost = int((L > cand).sum())
        print(f"           MAX_SRC={cand:<4} truncates {lost:>6,} ({lost/len(L)*100:5.2f}%)")

Lt = np.array([len(tok(s)["input_ids"]) for s in data["train"]["tagged"][:sample]])
Le = np.array([len(tok(text_target=s)["input_ids"]) for s in data["train"]["english"][:sample]])
MAX_SRC = 128
while (Lt > MAX_SRC).mean() > 0.001 and MAX_SRC < 256:
    MAX_SRC += 64
MAX_TGT = 128
while (Le > MAX_TGT).mean() > 0.001 and MAX_TGT < 256:
    MAX_TGT += 64
print(f"\ntarget   mean {Le.mean():5.1f}  99th {np.percentile(Le,99):3.0f}  max {Le.max():3.0f}")
print(f"\nMAX_SRC = {MAX_SRC}   MAX_TGT = {MAX_TGT}")

plain    mean  25.5  median  23  99th  60  max  92
           MAX_SRC=128  truncates      0 ( 0.00%)
           MAX_SRC=192  truncates      0 ( 0.00%)
           MAX_SRC=256  truncates      0 ( 0.00%)
tagged   mean  39.2  median  36  99th  83  max 132
           MAX_SRC=128  truncates      3 ( 0.01%)
           MAX_SRC=192  truncates      0 ( 0.00%)
           MAX_SRC=256  truncates      0 ( 0.00%)

target   mean  15.9  99th  37  max  63

MAX_SRC = 128   MAX_TGT = 128


## Cell 4: Dataset and configuration

In [9]:
from torch.utils.data import Dataset

class MTDataset(Dataset):
    def __init__(self, src, tgt, tokenizer, max_src, max_tgt):
        assert len(src) == len(tgt)
        self.src, self.tgt = src, tgt
        self.tok, self.max_src, self.max_tgt = tokenizer, max_src, max_tgt

    def __len__(self):
        return len(self.src)

    def __getitem__(self, i):
        # No padding here. DataCollatorForSeq2Seq pads each batch to its own
        # longest member and masks label padding with -100. Padding everything
        # to max_length would make ~70% of each batch padding, since the mean
        # source is around 39 tokens against a limit of 128.
        m = self.tok(self.src[i], max_length=self.max_src, truncation=True)
        t = self.tok(text_target=self.tgt[i], max_length=self.max_tgt,
                     truncation=True)
        return {"input_ids": m["input_ids"],
                "attention_mask": m["attention_mask"],
                "labels": t["input_ids"]}

if VRAM >= 10:
    BATCH, ACCUM = 32, 2
else:
    BATCH, ACCUM = 16, 4

EPOCHS, LR, WARMUP, BEAM = 3, 5e-5, 1000, 4
print(f"batch {BATCH} x accum {ACCUM} = effective {BATCH*ACCUM}   (original run: 64)")
print(f"epochs {EPOCHS}  lr {LR}  warmup {WARMUP}  beam {BEAM}")
print(f"steps/epoch {len(data['train']['plain'])//(BATCH*ACCUM):,}")

batch 32 x accum 2 = effective 64   (original run: 64)
epochs 3  lr 5e-05  warmup 1000  beam 4
steps/epoch 6,204


## Cell 5: Training routine

In [10]:
from transformers import (AutoModelForSeq2SeqLM, Seq2SeqTrainer,
                          Seq2SeqTrainingArguments, DataCollatorForSeq2Seq)

def train_system(variant, out_name):
    """variant: 'plain' or 'tagged'."""
    print(f"\n{'='*62}\n  {out_name}   source = {variant}\n{'='*62}")
    tokenizer = AutoTokenizer.from_pretrained(MT_MODEL)
    model = AutoModelForSeq2SeqLM.from_pretrained(MT_MODEL)
    print(f"parameters: {model.num_parameters()/1e6:.1f}M")

    tr = MTDataset(data["train"][variant], data["train"]["english"],
                   tokenizer, MAX_SRC, MAX_TGT)
    va = MTDataset(data["val"][variant], data["val"]["english"],
                   tokenizer, MAX_SRC, MAX_TGT)

    args = Seq2SeqTrainingArguments(
        output_dir=str(MODELS / f"_{out_name}_ckpt"),
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=LR,
        per_device_train_batch_size=BATCH,
        per_device_eval_batch_size=BATCH,
        gradient_accumulation_steps=ACCUM,
        num_train_epochs=EPOCHS,
        weight_decay=0.01,
        warmup_steps=WARMUP,
        fp16=True,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        save_total_limit=1,
        logging_steps=500,
        dataloader_num_workers=0,
        group_by_length=True,
        predict_with_generate=False,
        report_to="none",
        seed=42,
    )

    trainer = Seq2SeqTrainer(
        model=model, args=args,
        train_dataset=tr, eval_dataset=va,
        data_collator=DataCollatorForSeq2Seq(tokenizer, model=model))

    t0 = time.time()
    result = trainer.train()
    hours = (time.time() - t0) / 3600

    dest = MODELS / out_name
    trainer.save_model(str(dest))
    tokenizer.save_pretrained(str(dest))

    hist = [{"epoch": round(h["epoch"], 2), "eval_loss": h["eval_loss"]}
            for h in trainer.state.log_history if "eval_loss" in h]
    json.dump({"variant": variant, "model": MT_MODEL, "hours": round(hours, 2),
               "max_src": MAX_SRC, "max_tgt": MAX_TGT, "batch": BATCH,
               "grad_accum": ACCUM, "epochs": EPOCHS, "lr": LR,
               "train_loss": result.training_loss, "history": hist},
              open(RES / f"{out_name}_training.json", "w"), indent=2)

    print(f"\nsaved -> {dest.relative_to(ROOT)}   ({hours:.2f} h)")
    for h in hist:
        print(f"  epoch {h['epoch']:<5} eval_loss {h['eval_loss']:.4f}")

    del model, trainer
    gc.collect(); torch.cuda.empty_cache()
    return hours

print("train_system defined.")

train_system defined.


## Cell 6: Baseline (plain source)

Run this and let it finish before starting Cell 7. The model is saved inside
the function, so the result survives even if the next cell fails.

In [11]:
h_base = train_system("plain", "mt_baseline_v2")


  mt_baseline_v2   source = plain


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


pytorch_model.bin:   0%|          | 0.00/310M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

parameters: 77.5M


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/310M [00:00<?, ?B/s]

Epoch,Training Loss,Validation Loss
1,0.841300,0.747797
2,0.672200,0.659371
3,0.583600,0.627423


C:\Users\Haulai\miniconda3\envs\tka\lib\site-packages\transformers\modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 6, 'bad_words_ids': [[64171]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].



saved -> models\mt_baseline_v2   (0.95 h)
  epoch 1.0   eval_loss 0.7478
  epoch 2.0   eval_loss 0.6594
  epoch 3.0   eval_loss 0.6274


## Cell 7: Marked source

In [12]:
h_mark = train_system("tagged", "mt_marked_v2")


  mt_marked_v2   source = tagged
parameters: 77.5M


Epoch,Training Loss,Validation Loss
1,0.852200,0.758815
2,0.681000,0.667717
3,0.590200,0.637133


C:\Users\Haulai\miniconda3\envs\tka\lib\site-packages\transformers\modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'max_length': 512, 'num_beams': 6, 'bad_words_ids': [[64171]]}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.encoder.embed_positions.weight', 'model.decoder.embed_tokens.weight', 'model.decoder.embed_positions.weight', 'lm_head.weight'].



saved -> models\mt_marked_v2   (1.02 h)
  epoch 1.0   eval_loss 0.7588
  epoch 2.0   eval_loss 0.6677
  epoch 3.0   eval_loss 0.6371


## Cell 8: Summary

In [13]:
summary = {}
for name in ("mt_baseline_v2", "mt_marked_v2"):
    p = RES / f"{name}_training.json"
    if p.exists():
        summary[name] = json.load(open(p))
        print(f"{name:<18}{summary[name]['hours']:>7.2f} h   "
              f"final eval_loss {summary[name]['history'][-1]['eval_loss']:.4f}")
    else:
        print(f"{name:<18}  not trained")

for name in ("mt_baseline_v2", "mt_marked_v2"):
    d = MODELS / name
    print(("  ok   " if (d / "config.json").exists() else "  MISS ") + name)

print("\nNext: 07_mt_evaluation.ipynb")

mt_baseline_v2       0.95 h   final eval_loss 0.6274
mt_marked_v2         1.02 h   final eval_loss 0.6371
  ok   mt_baseline_v2
  ok   mt_marked_v2

Next: 07_mt_evaluation.ipynb
